Пример дискретно‑событийной модели потока зерна на элеватор на Python (одна смена, экспоненциальные интервалы прибытия, нормальное время разгрузки).

In [1]:
import heapq
import random
import statistics as stats

# ------------------------
# ПАРАМЕТРЫ МОДЕЛИ
# ------------------------

T_SIM = 8 * 60          # длительность моделирования, минут (8 часов)
S = 2                   # число приемных пунктов (ям)
MEAN_INTERARRIVAL = 12  # средний интервал между прибытиями, минут
MEAN_SERVICE = 20       # среднее время разгрузки, минут
STD_SERVICE = 5         # ст. отклонение времени разгрузки, минут

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------

def gen_interarrival():
    """Экспоненциальное распределение интервала между машинами."""
    # средняя величина = MEAN_INTERARRIVAL
    lamb = 1.0 / MEAN_INTERARRIVAL
    return random.expovariate(lamb)

def gen_service_time():
    """Нормальное распределение времени разгрузки (ограничиваем снизу)."""
    while True:
        x = random.gauss(MEAN_SERVICE, STD_SERVICE)
        if x > 1:   # минимальное время разгрузки 1 мин
            return x

# ------------------------
# СТРУКТУРЫ ДАННЫХ МОДЕЛИ
# ------------------------

# Очередь событий (минимальная куча)
# Элемент: (time, type, payload_dict)
event_queue = []

# Очередь автомобилей: список словарей с полями (id, arrival_time)
waiting_queue = []

# Состояние ресурсов
num_busy = 0            # сколько приемных пунктов занято сейчас

# Статистика
cars_arrived = 0
cars_served = 0
wait_times = []         # времена ожидания в очереди
queue_lengths = []      # снимки длины очереди
max_queue_len = 0

busy_time = 0.0         # интеграл "занятости" всех пунктов во времени
last_event_time = 0.0   # время последнего обработанного события

# ------------------------
# ИНИЦИАЛИЗАЦИЯ
# ------------------------

# Первое прибытие в момент 0
heapq.heappush(event_queue, (0.0, "ARRIVAL", {"id": 0}))
cars_arrived += 1

# ------------------------
# ОСНОВНОЙ ЦИКЛ ДИСКРЕТНО-СОБЫТИЙНОЙ МОДЕЛИ
# ------------------------

while event_queue:
    t, etype, payload = heapq.heappop(event_queue)

    if t > T_SIM:
        # Не моделируем события после окончания смены
        break

    # обновим интегральную занятость ресурсов
    dt = t - last_event_time
    if dt > 0:
        busy_time += num_busy * dt
    last_event_time = t

    # снимок длины очереди
    q_len = len(waiting_queue)
    queue_lengths.append(q_len)
    if q_len > max_queue_len:
        max_queue_len = q_len

    if etype == "ARRIVAL":
        # Прибытие новой машины
        car_id = payload["id"]
        arrival_time = t

        # Планируем следующее прибытие (если не вышли за T_SIM)
        next_arrival_time = t + gen_interarrival()
        if next_arrival_time <= T_SIM:
            cars_arrived += 1
            heapq.heappush(event_queue, (next_arrival_time, "ARRIVAL",
                                         {"id": cars_arrived - 1}))

        # Проверяем наличие свободного приемного пункта
        if num_busy < S:
            # Можем обслуживать сразу
            num_busy += 1
            service_time = gen_service_time()
            depart_time = t + service_time
            heapq.heappush(event_queue, (depart_time, "DEPARTURE",
                                         {"id": car_id,
                                          "arrival_time": arrival_time}))
            # ожидание в очереди = 0
            wait_times.append(0.0)
        else:
            # Встаем в очередь
            waiting_queue.append({"id": car_id, "arrival_time": arrival_time})

    elif etype == "DEPARTURE":
        # Завершение разгрузки
        cars_served += 1

        if waiting_queue:
            # Берем следующую машину из очереди (FIFO)
            next_car = waiting_queue.pop(0)
            w = t - next_car["arrival_time"]
            wait_times.append(w)

            service_time = gen_service_time()
            depart_time = t + service_time
            heapq.heappush(event_queue, (depart_time, "DEPARTURE",
                                         {"id": next_car["id"],
                                          "arrival_time": next_car["arrival_time"]}))
            # num_busy не меняется (одна машина уехала, другая заехала)
        else:
            # Больше никого не ждёт — приемный пункт становится свободным
            num_busy -= 1

# ------------------------
# ОБРАБОТКА РЕЗУЛЬТАТОВ
# ------------------------

sim_time = last_event_time
avg_wait = stats.mean(wait_times) if wait_times else 0.0
avg_queue = stats.mean(queue_lengths) if queue_lengths else 0.0
utilization = busy_time / (sim_time * S) if sim_time > 0 else 0.0

print(f"Смена моделировалась до времени: {sim_time:.1f} мин")
print(f"Число прибывших машин: {cars_arrived}")
print(f"Число обслуженных машин: {cars_served}")
print(f"Среднее время ожидания в очереди: {avg_wait:.2f} мин")
print(f"Средняя длина очереди: {avg_queue:.2f}")
print(f"Максимальная длина очереди: {max_queue_len}")
print(f"Использование приемных пунктов (доля времени занятости): {utilization:.3f}")


Смена моделировалась до времени: 477.7 мин
Число прибывших машин: 39
Число обслуженных машин: 39
Среднее время ожидания в очереди: 19.46 мин
Средняя длина очереди: 1.88
Максимальная длина очереди: 6
Использование приемных пунктов (доля времени занятости): 0.814
